In [1]:
!pip install faiss-cpu
!pip install faiss-gpu

In [5]:
import torch
import numpy as np
from datasets import Dataset
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используем устройство: {device}")

# ==========================================
# 1. ПОДГОТОВКА ДАННЫХ И ВЕКТОРНАЯ БАЗА (datasets + FAISS)
# ==========================================
# Имитируем базу знаний МТС EnergyTool
knowledge_base = {
    "title": ["Умный свет", "Мониторинг станков", "Энергоаудит", "Резервное питание"],
    "content": [
        "Тариф 'Умный свет': датчики движения и LED-лампы. Экономия до 40% на освещении складов.",
        "Услуга 'Мониторинг': IoT-датчики на станки. Показывает холостой ход и перерасход энергии в цехах.",
        "Услуга 'Энергоаудит': выезд инженера для поиска утечек тепла и неэффективного оборудования.",
        "Тариф 'Резерв': аренда промышленных генераторов и ИБП для защиты серверов от скачков напряжения."
    ]
}
# Превращаем словарь в объект Dataset
dataset = Dataset.from_dict(knowledge_base)

# TODO 1: Инициализируй pipeline для извлечения фичей (feature-extraction).
embedder = pipeline(
    task="feature-extraction",
    model="cointegrated/rubert-tiny2",
    device=device
)

def get_embeddings(batch):
    # TODO 2: Получи эмбеддинги для батча текстов (batch["content"]).
    vectors = [embed[0][0] for embed in embedder(batch["content"])] # Напиши логику извлечения
    return {"embeddings": vectors}

print("Векторизуем базу знаний...")
# TODO 3: Примени функцию get_embeddings к датасету с помощью метода .map().
vectorized_dataset = dataset.map(
    get_embeddings,
    batched=True,
    batch_size=128
)

# TODO 4: Построй FAISS индекс по колонке "embeddings".
vectorized_dataset.add_faiss_index(column="embeddings")

# ==========================================
# 2. ПОИСК (Retrieval)
# ==========================================
user_query = "У нас на складе огромные счета за электричество из-за постоянно горящих ламп. Что у вас есть?"
print(f"\nЗапрос клиента: {user_query}")

# TODO 5: Векторизуй user_query с помощью embedder (получи один вектор).
query_list = embedder(user_query)[0][0]
query_vector = np.array(query_list, dtype=np.float32)

# TODO 6: Найди 1 самый релевантный документ в vectorized_dataset с помощью .get_nearest_examples().
# Извлеки текст найденного решения (content).
score, retrieved_context = vectorized_dataset.get_nearest_examples(
    "embeddings",
    query_vector,
    k=1
)
print(f"Найденный контекст в базе: {retrieved_context['content'][0]}")


# ==========================================
# 3. ГЕНЕРАЦИЯ ОТВЕТА (LLM + Chat Templates)
# ==========================================
# Для скорости на тестах возьмем крошечную LLM (Qwen 1.5B или TinyLlama)
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

print("\nЗагружаем LLM (может занять время)...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16
).to(device)

# TODO 7: Сформируй список словарей (messages) с ролями 'system' и 'user'.
messages = [
    {
        "role": "system",
        "content": "Ты — ИИ-консультант МТС EnergyTool. Отвечай кратко, вежливо и предлагай тарифы из контекста."
    },

    {
        "role": "user",
        "content": f"Вопрос клиента: {user_query} Наиболее подходящее предложение: {retrieved_context['content'][0]}."
    }
]

# TODO 8: Примени chat template к messages.
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(device)

print("Генерируем ответ...")
with torch.no_grad():
    # TODO 9: Сгенерируй ответ с помощью model.generate().
    generated_ids = model.generate(
        **inputs, # расспаковываем прямиком в атрибуты
        max_new_tokens=150,
        temperature=0.3
    )

# Извлекаем только новые токены (отрезаем промпт)
new_tokens = generated_ids[0][inputs["input_ids"].shape[1]:]

# TODO 10: Декодируй новые токены с помощью tokenizer.decode().
final_answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print(f"\nОтвет ИИ-консультанта:\n{final_answer}")

Используем устройство: cuda


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Векторизуем базу знаний...


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

  0%|          | 0/1 [00:00<?, ?it/s]


Запрос клиента: У нас на складе огромные счета за электричество из-за постоянно горящих ламп. Что у вас есть?
Найденный контекст в базе: Тариф 'Резерв': аренда промышленных генераторов и ИБП для защиты серверов от скачков напряжения.

Загружаем LLM (может занять время)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Генерируем ответ...

Ответ ИИ-консультанта:
Мне кажется, вы имеете в виду "Резервный энергетический план". Этот тариф предлагает арендовать промышленные генераторы и индивидуальную батарейную систему поддержания питания (IBP) для защиты серверов от скачков напряжения. Это может быть полезным решением для вашего случая с горящими лампами.
